In [1]:
!pip install pycocotools

In [2]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def evaluate_coco(gt_json_path, pred_json_path, iou_type="bbox"):
    """
    Evaluate COCO-format predictions against ground truth.

    Parameters
    ----------
    gt_json_path : str
        Path to COCO ground truth annotation file
    pred_json_path : str
        Path to COCO detection results file
    iou_type : str
        One of: "bbox", "segm", "keypoints"
    """

    print(f"Loading ground truth: {gt_json_path}")
    coco_gt = COCO(gt_json_path)

    print(f"Loading predictions: {pred_json_path}")
    coco_dt = coco_gt.loadRes(pred_json_path)

    print(f"\nRunning COCO evaluation (iou_type='{iou_type}')...\n")
    coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval

In [4]:
gt_path = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json"   # Ground truth COCO file
pred_path = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_annotations\test_fold_0_results.json"                       # Your prediction file

coco_eval = evaluate_coco(gt_path, pred_path, iou_type="bbox")

Loading ground truth: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading predictions: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_annotations\test_fold_0_results.json
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!

Running COCO evaluation (iou_type='bbox')...

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.32s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.217
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.384
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.225
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.182
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.197
 Averag

In [6]:
gt_path = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json"   # Ground truth COCO file
pred_path = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_0_predictions.json"                       # Your prediction file

coco_eval = evaluate_coco(gt_path, pred_path, iou_type="bbox")

Loading ground truth: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Loading predictions: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_0_predictions.json
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!

Running COCO evaluation (iou_type='bbox')...

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.33s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.079
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.035
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=

## For each fold SAM 3

In [5]:
import os
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def evaluate_coco(gt_json_path, pred_json_path, iou_type="bbox"):
    print(f"Loading ground truth: {gt_json_path}")
    coco_gt = COCO(gt_json_path)

    print(f"Loading predictions: {pred_json_path}")
    coco_dt = coco_gt.loadRes(pred_json_path)

    print(f"\nRunning COCO evaluation (iou_type='{iou_type}')...\n")
    coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval


GT_TEMPLATE = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_{}.json"
PRED_TEMPLATE = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_results\test_fold_{}_results.json"

all_results = {}

for fold in range(5):  # folds 0,1,2,3,4
    print("\n" + "=" * 60)
    print(f"🚀 Evaluating FOLD {fold}")
    print("=" * 60)

    gt_path = GT_TEMPLATE.format(fold)
    pred_path = PRED_TEMPLATE.format(fold)

    if not os.path.exists(gt_path):
        print(f"❌ Missing GT file for fold {fold}: {gt_path}")
        continue
    if not os.path.exists(pred_path):
        print(f"❌ Missing prediction file for fold {fold}: {pred_path}")
        continue

    coco_eval = evaluate_coco(gt_path, pred_path, iou_type="bbox")

    # Store main COCO metrics (AP, AP50, etc.)
    all_results[fold] = coco_eval.stats  # numpy array with 12 standard metrics


print("\n" + "#" * 60)
print("📊 Cross-fold summary")
print("#" * 60)

for fold, stats in all_results.items():
    print(f"Fold {fold} AP @[IoU=0.50:0.95]: {stats[0]:.4f}")

print("\n" + "#" * 60)
print("📊 Cross-fold summary (Mean AP per condition)")
print("#" * 60)

import numpy as np

if all_results:
    stats_array = np.array(list(all_results.values()))  # shape: [num_folds, 12]

    mean_stats = stats_array.mean(axis=0)

    print(f"Mean AP @[IoU=0.50:0.95]: {mean_stats[0]:.4f}")
    print(f"Mean AP @[IoU=0.50]     : {mean_stats[1]:.4f}")
    print(f"Mean AP @[IoU=0.75]     : {mean_stats[2]:.4f}")
    print(f"Mean AP (small objects) : {mean_stats[3]:.4f}")
    print(f"Mean AP (medium objects): {mean_stats[4]:.4f}")
    print(f"Mean AP (large objects) : {mean_stats[5]:.4f}")

    print("\nPer-fold AP @[IoU=0.50:0.95]:")
    for fold, stats in all_results.items():
        print(f"  Fold {fold}: {stats[0]:.4f}")
else:
    print("No valid folds were evaluated.")




🚀 Evaluating FOLD 0
Loading ground truth: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading predictions: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\SAM3\coco_results\test_fold_0_results.json
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!

Running COCO evaluation (iou_type='bbox')...

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.28s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.217
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.384
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.225
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.182
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 

## QWEN 3 VL 2B

In [10]:
import os
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def evaluate_coco(gt_json_path, pred_json_path, iou_type="bbox"):
    print(f"Loading ground truth: {gt_json_path}")
    coco_gt = COCO(gt_json_path)

    print(f"Loading predictions: {pred_json_path}")
    coco_dt = coco_gt.loadRes(pred_json_path)

    print(f"\nRunning COCO evaluation (iou_type='{iou_type}')...\n")
    coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval


GT_TEMPLATE = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_{}.json"
PRED_TEMPLATE = r"C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_{}_results.json"

all_results = {}

for fold in range(5):  # folds 0,1,2,3,4
    print("\n" + "=" * 60)
    print(f"🚀 Evaluating FOLD {fold}")
    print("=" * 60)

    gt_path = GT_TEMPLATE.format(fold)
    pred_path = PRED_TEMPLATE.format(fold)

    if not os.path.exists(gt_path):
        print(f"❌ Missing GT file for fold {fold}: {gt_path}")
        continue
    if not os.path.exists(pred_path):
        print(f"❌ Missing prediction file for fold {fold}: {pred_path}")
        continue

    coco_eval = evaluate_coco(gt_path, pred_path, iou_type="bbox")

    # Store main COCO metrics (AP, AP50, etc.)
    all_results[fold] = coco_eval.stats  # numpy array with 12 standard metrics


print("\n" + "#" * 60)
print("📊 Cross-fold summary")
print("#" * 60)

for fold, stats in all_results.items():
    print(f"Fold {fold} AP @[IoU=0.50:0.95]: {stats[0]:.4f}")

print("\n" + "#" * 60)
print("📊 Cross-fold summary (Mean AP per condition)")
print("#" * 60)

import numpy as np

if all_results:
    stats_array = np.array(list(all_results.values()))  # shape: [num_folds, 12]

    mean_stats = stats_array.mean(axis=0)

    print(f"Mean AP @[IoU=0.50:0.95]: {mean_stats[0]:.4f}")
    print(f"Mean AP @[IoU=0.50]     : {mean_stats[1]:.4f}")
    print(f"Mean AP @[IoU=0.75]     : {mean_stats[2]:.4f}")
    print(f"Mean AP (small objects) : {mean_stats[3]:.4f}")
    print(f"Mean AP (medium objects): {mean_stats[4]:.4f}")
    print(f"Mean AP (large objects) : {mean_stats[5]:.4f}")

    print("\nPer-fold AP @[IoU=0.50:0.95]:")
    for fold, stats in all_results.items():
        print(f"  Fold {fold}: {stats[0]:.4f}")
else:
    print("No valid folds were evaluated.")



🚀 Evaluating FOLD 0
Loading ground truth: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\src\dataset\video_pineapple\seeds\test_fold_0.json
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading predictions: C:\Users\dnnxl\Documents\GitHub\vlm-od-agriculture\outputs\Qwen3-VL-32B-Instruct\coco_predictions\test_fold_0_results.json
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!

Running COCO evaluation (iou_type='bbox')...

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.30s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.079
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.035
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=